In [ ]:
# ============================================================
# CLIMATE IMPACT METRICS (LCA-BASED)
# ============================================================

# ----------------------------
# 1. IMPORT LIBRARIES
# ----------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------
# 2. CREATE OUTPUT FOLDER
# ----------------------------
os.makedirs("figures", exist_ok=True)


# ----------------------------
# 3. LOAD DATA
# ----------------------------
file_path = "example_dataset.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 4. FUNCTION — CLIMATE IMPACT METRICS
# ============================================================

def compute_lca_metrics(
    df,
    leak_col="CO2 Total Leakage Mass",
    secure_col="CO2 Total Secure Mass",
    leakage_fraction_col="Leakage Fraction"
):
    """
    Computes simple climate-impact metrics based on GWP100.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.
    leak_col : str
        Column containing leaked CO2 mass.
    secure_col : str
        Column containing securely stored CO2 mass.
    leakage_fraction_col : str
        Column containing leakage fraction.

    Returns
    -------
    pandas.DataFrame
        Dataset with added LCA-based metrics:
        - CO2_eq_leakage
        - CO2_eq_avoided
        - Net_climate_benefit
    """

    df = df.copy()

    GWP100 = 1.0  # CO2 reference value

    # Climate impact of leaked CO2
    df["CO2_eq_leakage"] = df[leak_col] * GWP100

    # Climate benefit of securely stored CO2
    df["CO2_eq_avoided"] = df[secure_col] * GWP100

    # Net climate benefit
    df["Net_climate_benefit"] = df["CO2_eq_avoided"] - df["CO2_eq_leakage"]

    return df


# ----------------------------
# 5. COMPUTE METRICS
# ----------------------------
df_lca = compute_lca_metrics(
    df,
    leak_col="CO2 Total Leakage Mass",
    secure_col="CO2 Total Secure Mass",
    leakage_fraction_col="Leakage Fraction"
)

print("\nLCA columns added:")
print(["CO2_eq_leakage", "CO2_eq_avoided", "Net_climate_benefit"])


# ============================================================
# 6. SCATTER / HEXBIN: LEAKAGE FRACTION VS CLIMATE IMPACT
# ============================================================

x = df_lca["Leakage Fraction"].values
y = df_lca["CO2_eq_leakage"].values

# Remove NaN or inf values
mask = np.isfinite(x) & np.isfinite(y)
x = x[mask]
y = y[mask]

# Only fit line if enough points exist
if len(x) >= 2:
    m, b = np.polyfit(x, y, 1)
    x_sorted = np.sort(x)
else:
    m, b = None, None
    x_sorted = x

plt.figure(figsize=(8, 5), dpi=300)

hb = plt.hexbin(
    x,
    y,
    gridsize=35,
    cmap="viridis",
    bins="log",
    mincnt=1,
    linewidths=0.2,
    alpha=0.95
)

if m is not None:
    plt.plot(
        x_sorted,
        m * x_sorted + b,
        color="black",
        linewidth=2.0,
        label="Linear trend"
    )

cbar = plt.colorbar(hb)
cbar.set_label("log(count)", fontsize=11)

plt.xlabel("Leakage Fraction", fontsize=13)
plt.ylabel("Climate Impact of Leakage (t CO₂-eq)", fontsize=13)
plt.title("Leakage Fraction vs Climate Impact", fontsize=15)

plt.grid(alpha=0.25, linestyle="--", linewidth=0.5)

if m is not None:
    plt.legend(frameon=False, fontsize=11)

plt.tight_layout()

plt.savefig(
    "figures/Figure_LCA_LeakageFraction_vs_ClimateImpact.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# 7. SCATTER: LEAKAGE FRACTION VS NET CLIMATE BENEFIT
# ============================================================

x2 = df_lca["Leakage Fraction"].values
y2 = df_lca["Net_climate_benefit"].values

mask2 = np.isfinite(x2) & np.isfinite(y2)
x2 = x2[mask2]
y2 = y2[mask2]

if len(x2) >= 2:
    m2, b2 = np.polyfit(x2, y2, 1)
    x2_sorted = np.sort(x2)
else:
    m2, b2 = None, None
    x2_sorted = x2

plt.figure(figsize=(8, 5), dpi=300)

plt.scatter(
    x2,
    y2,
    s=35,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.4
)

if m2 is not None:
    plt.plot(
        x2_sorted,
        m2 * x2_sorted + b2,
        color="black",
        linewidth=2.0,
        label="Linear trend"
    )

plt.axhline(0, color="gray", linestyle="--", linewidth=1)

plt.xlabel("Leakage Fraction", fontsize=13)
plt.ylabel("Net Climate Benefit (t CO₂-eq)", fontsize=13)
plt.title("Leakage Fraction vs Net Climate Benefit", fontsize=15)

plt.grid(alpha=0.25, linestyle="--", linewidth=0.5)

if m2 is not None:
    plt.legend(frameon=False, fontsize=11)

plt.tight_layout()

plt.savefig(
    "figures/Figure_LCA_LeakageFraction_vs_NetClimateBenefit.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# 8. SUMMARY TABLE
# ============================================================

summary = df_lca[
    ["CO2_eq_leakage", "CO2_eq_avoided", "Net_climate_benefit", "Leakage Fraction"]
].describe()

print("\n=== Summary Statistics ===")
print(summary)


# ============================================================
# 9. SAVE OUTPUT DATASET
# ============================================================

df_lca.to_excel("LCA_metrics_output.xlsx", index=False)
print("\nSaved file: LCA_metrics_output.xlsx")